# Testing Biobert

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel


/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# BioBERT as a Feature Extractor (outputing embeddings)

- it is a pretrained encoder

- for NER we must add a NER head and fientune it. 

In [ ]:

# Load BioBERT tokenizer (handles tokenization + WordPiece)
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

# Load BioBERT model (returns hidden states, not task-specific predictions)
model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of model parameters: {num_params:,}")
# Put model in evaluation mode (disables dropout)
model.eval()

# Example biomedical text
text = "The patient was treated with aspirin for myocardial infarction."

# Tokenize text:
# - return_tensors="pt" returns PyTorch tensors
# - padding/truncation are useful for batch processing
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# Disable gradient computation (inference only)
with torch.no_grad():
    outputs = model(**inputs)

# outputs.last_hidden_state shape:
# (batch_size, sequence_length, hidden_size)
last_hidden_state = outputs.last_hidden_state

# CLS token embedding (often used for sentence-level tasks)
cls_embedding = last_hidden_state[:, 0, :]

print("Last hidden state shape:", last_hidden_state.shape)
print("CLS embedding shape:", cls_embedding.shape)


Number of model parameters: 108,310,272
Last hidden state shape: torch.Size([1, 19, 768])
CLS embedding shape: torch.Size([1, 768])


# Preparing for Training

In [16]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForTokenClassification
)
# local import
from config import settings

dataset = load_dataset(settings.HUGGINGFACE_REPO_ID)
# NOTE that the actual labels that will be used for training are under the column: "token_label_ids"
dataset = dataset.rename_column("token_label_ids", "labels")

# Download and load id2label.json from the hub
id2label_path = hf_hub_download(
    repo_id=settings.HUGGINGFACE_REPO_ID,
    filename="id2label.json",
    repo_type="dataset"
)
# Download and load label2id.json from the hub
label2id_path = hf_hub_download(
    repo_id=settings.HUGGINGFACE_REPO_ID,
    filename="label2id.json",
    repo_type="dataset"
)
with open(id2label_path, "r") as f:
    id2label = json.load(f)
with open(label2id_path, "r") as f:
    label2id = json.load(f)

# Number of labels 
num_labels = len(id2label)
# convert label keys to integers
id2label = {int(k): v for k, v in id2label.items()}

# model name
MODEL_NAME = "dmis-lab/biobert-v1.1"
# config = AutoConfig.from_pretrained(
#     MODEL_NAME,
#     num_labels=num_labels,
#     id2label=id2label,
#     label2id=label2id
# )

tokenizer_biobert = AutoTokenizer.from_pretrained(MODEL_NAME)
# 3. Load model with token-classification head
model = AutoModelForTokenClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
print("Distill bioBERT:\n", tokenizer_biobert)
 # ============
MODEL_NAME =  "distilbert-base-uncased"
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

tokenizer_distill = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Distill TOkenizer:\n", tokenizer_distill)
# 3. Load model with token-classification head
model_distill = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

print("Are the tokenizers the same? ", tokenizer_biobert == tokenizer_distill)





Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Distill bioBERT:
 BertTokenizerFast(name_or_path='dmis-lab/biobert-v1.1', vocab_size=28996, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Distill TOkenizer:
 DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
Are the tokenizers the same?  False


In [ ]:
# Distill Bert configuration
# model name
MODEL_NAME =  "distilbert-base-uncased"
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

tokenizer_distill = AutoTokenizer.from_pretrained(MODEL_NAME)
# 3. Load model with token-classification head
model_distill = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config
)
print(f"About the {MODEL_NAME} tokenizer: {tokenizer}")

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


About the distilbert-base-uncased tokenizer: BertTokenizerFast(name_or_path='dmis-lab/biobert-v1.1', vocab_size=28996, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
